# Libraries

In [0]:
%pip install numpy==1.26 pymc-marketing==0.15
%restart_python

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import arviz as az
import mlflow

from pymc_marketing.mmm import MMM, GeometricAdstock, LogisticSaturation
from pymc_marketing.mmm.transformers import geometric_adstock, logistic_saturation
from pymc_marketing.prior import Prior
from sklearn.metrics import r2_score, mean_squared_error as mse, mean_absolute_percentage_error as mape
from sklearn.preprocessing import *
from sklearn.linear_model import *
from sklearn.model_selection import *
from scipy.optimize import curve_fit

mlflow.autolog(disable=True)

# Data

## Reading

In [0]:
path = '/Volumes/training_feature_development/marketing_mix_modelling/data_v1'

# Reading data
df1 = pd.read_csv(f'{path}/model_data/exh_awareness_mmm.csv')
df2 = pd.read_csv(f'{path}/model_data/exh_consideration_mmm.csv')
df3 = pd.read_csv(f'{path}/model_data/exh_conversion_mmm.csv')

df1['week'] = pd.to_datetime(df1['week'])
df2['week'] = pd.to_datetime(df2['week'])
df3['week'] = pd.to_datetime(df3['week'])

# Benchmark Linear Model

## Fitting

In [0]:
# Defining variables
X1 = df1[df1.columns[1:-3]]
X2 = df2[df2.columns[1:-3]]
X3 = df3[df3.columns[1:-3]]
y = df1['attendance']

# Transforming variables
for i in [X1, X2, X3]:
    for j in list(i.columns)[6:]:
        i[j] = MinMaxScaler().fit_transform(i[[j]]).flatten()
        i[j] = geometric_adstock(i[j], alpha=0.5, l_max=4, normalize=True).eval().flatten()
        i[j] = logistic_saturation(i[j], lam=5).eval().flatten()

In [0]:
# Fitting model
model1 = LinearRegression().fit(X1, y)
model2 = LinearRegression().fit(X2, y)
model3 = LinearRegression().fit(X3, y)

y_pred1 = pd.Series(model1.predict(X1), index=df1.index)
y_pred2 = pd.Series(model2.predict(X2), index=df1.index)
y_pred3 = pd.Series(model3.predict(X3), index=df1.index)

In [0]:
fig, ax = plt.subplots(figsize=(9, 5), dpi=240)

# Plotting model
plt.plot(y_pred1, color='tab:blue', label='Predicted Attendance (Awareness)')
plt.plot(y_pred2, color='tab:orange', label='Predicted Attendance (Consideration)')
plt.plot(y_pred3, color='tab:green', label='Predicted Attendance (Conversion)')
plt.plot(y, color='grey', alpha=0.5, label='Actual Attendance')

# Changing aesthetics
plt.ylim(0, 3*10**6)
plt.xlabel('Week Number')
plt.ylabel('Attendance')
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.grid(axis='y', alpha=0.25)
plt.legend()

plt.show()

## Evaluation

In [0]:
# Evaluating model
m1 = [r2_score(y, y_pred1), r2_score(y, y_pred2), r2_score(y, y_pred3)]
m2 = [np.sqrt(mse(y, y_pred1)), np.sqrt(mse(y, y_pred2)), np.sqrt(mse(y, y_pred3))]
m3 = [mape(y, y_pred1), mape(y, y_pred2), mape(y, y_pred3)]

print('R2 =', np.round(m1, 3))
print('RMSE =', np.round(m2))
print('MAPE =', np.round(m3, 3))

In [0]:
# summary = pd.DataFrame({
#     'Feature': X1.columns,
#     'Coefficient': np.round(model1.coef_, 0)
# })
# print(summary.sort_values('Coefficient', ascending=False).reset_index(drop=True))

# Marketing Mix Model (Awareness)

In [0]:
# Defining variables
X1 = df1[df1.columns[:-3]]
y = df1['attendance']

## Fitting

In [0]:
# Setting assumptions
custom_config = {
    # Non-negative baseline sales
    'intercept': Prior('Exponential', lam=1),
    # Setting a non-negative control effect
    'gamma_control': Prior('HalfNormal', sigma=Prior('Gamma', alpha=4, beta=1)),
    # Setting an average retention rate of 40% 
    'adstock_alpha': Prior('Beta', alpha=2, beta=3),
    # Setting a moderate ceiling for spend
    'saturation_lam': Prior('Gamma', alpha=3, beta=2)
}

mmm1 = MMM(model_config=custom_config,
          date_column='week',
          channel_columns=X1.columns[7:],
          control_columns=X1.columns[1:7],
          # Maximum effect over 3 weeks
          adstock=GeometricAdstock(l_max=3),
          saturation=LogisticSaturation())

mmm1.fit(X1, y, chains=2, random_seed=12, target_accept=0.95)

mmm1.sample_prior_predictive(X1, extend_idata=True, combined=True)
mmm1.sample_posterior_predictive(X1, extend_idata=True, combined=True)

y_pred1 = mmm1.predict(X1)

In [0]:
fig, ax = plt.subplots(figsize=(9, 5), dpi=240)

# Plotting model
plt.plot(X1['week'], y_pred1, color='tab:blue', label='Predicted Attendance')
plt.plot(X1['week'], y, color='grey', alpha=0.5, label='Actual Attendance')

# Changing aesthetics
plt.ylim(0, 3*10**6)
plt.xlabel('Week')
plt.ylabel('Attendance')
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper right')

plt.show()

## Diagnostics

In [0]:
# Checking divergences in the simulations
if mmm1.idata['sample_stats']['diverging'].sum().item() == 0:
    print('No divergences found')
else:
    print('Divergences found')

In [0]:
# Plotting contribution proportions
mmm1.plot_waterfall_components_decomposition(figsize=(18, 10), dpi=120);
plt.grid(axis='y', alpha=0.25)

## Evaluation

In [0]:
# Evaluating model
m1 = r2_score(y, y_pred1)
m2 = np.sqrt(mse(y, y_pred1))
m3 = mape(y, y_pred1)

print('R2 =', np.round(m1, 3))
print('RMSE =', np.round(m2))
print('MAPE =', np.round(m3, 3))

# Marketing Mix Model (Consideration)

In [0]:
# Defining variables
X2 = df2[df2.columns[:-3]]
y = df2['attendance']

## Fitting

In [0]:
# Setting assumptions
custom_config = {
    # Non-negative baseline sales
    'intercept': Prior('Exponential', lam=1),
    # Setting a non-negative control effect
    'gamma_control': Prior('HalfNormal', sigma=Prior('Gamma', alpha=4, beta=1)),
    # Setting an average retention rate of 40% 
    'adstock_alpha': Prior('Beta', alpha=2, beta=3),
    # Setting a moderate ceiling for spend
    'saturation_lam': Prior('Gamma', alpha=4, beta=1)
}

mmm2 = MMM(model_config=custom_config,
          date_column='week',
          channel_columns=X2.columns[7:],
          control_columns=X2.columns[1:7],
          # Maximum effect over 3 weeks
          adstock=GeometricAdstock(l_max=3),
          saturation=LogisticSaturation())

mmm2.fit(X2, y, chains=2, random_seed=12, target_accept=0.95)

mmm2.sample_prior_predictive(X2, extend_idata=True, combined=True)
mmm2.sample_posterior_predictive(X2, extend_idata=True, combined=True)

y_pred2 = mmm2.predict(X2)

In [0]:
fig, ax = plt.subplots(figsize=(9, 5), dpi=240)

# Plotting model
plt.plot(X2['week'], y_pred2, color='tab:orange', label='Predicted Attendance')
plt.plot(X2['week'], y, color='grey', alpha=0.5, label='Actual Attendance')

# Changing aesthetics
plt.ylim(0, 3*10**6)
plt.xlabel('Week')
plt.ylabel('Attendance')
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper right')

plt.show()

## Diagnostics

In [0]:
# Checking divergences in the simulations
if mmm2.idata['sample_stats']['diverging'].sum().item() == 0:
    print('No divergences found')
else:
    print('Divergences found')

In [0]:
# Plotting contribution proportions
mmm2.plot_waterfall_components_decomposition(figsize=(18, 10), dpi=120);
plt.grid(axis='y', alpha=0.25)

## Evaluation

In [0]:
# Evaluating model
m1 = r2_score(y, y_pred2)
m2 = np.sqrt(mse(y, y_pred2))
m3 = mape(y, y_pred2)

print('R2 =', np.round(m1, 3))
print('RMSE =', np.round(m2))
print('MAPE =', np.round(m3, 3))

# Marketing Mix Model (Conversion)

In [0]:
# Defining variables
X3 = df3[df3.columns[:-3]]
y = df3['attendance']

## Fitting

In [0]:
# Setting assumptions
custom_config = {
    # Non-negative baseline sales
    'intercept': Prior('Exponential', lam=1),
    # Setting a non-negative control effect
    'gamma_control': Prior('HalfNormal', sigma=Prior('Gamma', alpha=4, beta=1)),
    # Setting an average retention rate of 40% 
    'adstock_alpha': Prior('Beta', alpha=2, beta=3),
    # Setting a moderate ceiling for spend
    'saturation_lam': Prior('Gamma', alpha=4, beta=1)
}

# Fitting model
mmm3 = MMM(model_config=custom_config,
          date_column='week',
          channel_columns=X3.columns[7:],
          control_columns=X3.columns[1:7],
          # Maximum effect over 4 weeks
          adstock=GeometricAdstock(l_max=3),
          saturation=LogisticSaturation())

mmm3.fit(X3, y, chains=2, random_seed=12, target_accept=0.95)

# Sampling prior and posterior distributions
mmm3.sample_prior_predictive(X3, extend_idata=True, combined=True)
mmm3.sample_posterior_predictive(X3, extend_idata=True, combined=True)

y_pred3 = mmm3.predict(X3)

In [0]:
fig, ax = plt.subplots(figsize=(9, 5), dpi=240)

# Plotting model
plt.plot(X3['week'], y_pred3, color='tab:green', label='Predicted Attendance')
plt.plot(X3['week'], y, color='grey', alpha=0.5, label='Actual Attendance')

# Changing aesthetics
plt.ylim(0, 3*10**6)
plt.xlabel('Week')
plt.ylabel('Attendance')
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper right')

plt.show()

## Diagnostics

In [0]:
# Checking divergences in the simulations
if mmm3.idata['sample_stats']['diverging'].sum().item() == 0:
    print('No divergences found')
else:
    print('Divergences found')

In [0]:
# Plotting contribution proportions
mmm3.plot_waterfall_components_decomposition(figsize=(18, 10), dpi=120);
plt.grid(axis='y', alpha=0.25)

## Evaluation

In [0]:
# Evaluating model
m1 = r2_score(y, y_pred3)
m2 = np.sqrt(mse(y, y_pred3))
m3 = mape(y, y_pred3)

print('R2 =', np.round(m1, 3))
print('RMSE =', np.round(m2))
print('MAPE =', np.round(m3, 3))

## Parameters and Deep-Dive

In [0]:
# Defining a function for plotting
def plot_contribution_curve(channel: str):
    fig = mmm3.plot_direct_contribution_curves(channels=[channel], show_fit=True, xlim_max=1.5, quantile_lower=0.5, quantile_upper=0.5)
    fig.set_size_inches((9, 5))
    fig.set_dpi(240)
    plt.gca().xaxis.set_major_formatter(mtick.EngFormatter(places=0, unit=''))
    plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(places=0, unit=''))
    plt.grid(alpha=0.25)

# Plotting contribution curves
plot_contribution_curve('meta_brand')
plot_contribution_curve('meta_event')
plot_contribution_curve('meta_film')

In [0]:
contributions_over_time = mmm3.compute_mean_contributions_over_time(
    original_scale=True
)

plt.figure(figsize=(9, 5), dpi=240)
plt.plot(contributions_over_time['meta_brand'], color='tab:blue', label='Meta Brand')
plt.plot(contributions_over_time['meta_event'], color='tab:orange', label='Meta Event')
plt.plot(contributions_over_time['meta_film'], color='tab:green', label='Meta Film')
plt.xlabel('Week')
plt.ylabel('Contribution')
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(places=0, unit=''))
plt.legend(loc='upper right')
plt.grid(axis='y', alpha=0.25)
plt.show()

## ROAS

In [0]:
# Plotting ROAS by channel
channel_contribution_original_scale = mmm3.compute_channel_contribution_original_scale()
spend_sum = X3[X3.columns[7:]].sum().to_numpy()

roas_samples = (
    channel_contribution_original_scale.sum(dim="date")
    / spend_sum[np.newaxis, np.newaxis, :]
)

fig, axes = plt.subplots(
    nrows=13, ncols=1, figsize=(14, 42), dpi=120, sharex=True, sharey=False, layout="constrained"
)
az.plot_posterior(roas_samples, ax=axes);

# Marketing Mix Model (LOB)

In [0]:
lob_df1 = df1[['week', 'summer', 'holiday', 'releases', 'mass_exh', 'mass_film', 'mass_store', 'pr_spend']].copy()
lob_df2 = df2[['week', 'summer', 'holiday', 'releases', 'mass_exh', 'mass_film', 'mass_store', 'pr_spend']].copy()
lob_df3 = df3[['week', 'summer', 'holiday', 'releases', 'mass_exh', 'mass_film', 'mass_store', 'pr_spend']].copy()

In [0]:
lob_df1['lob_brand'] = df1[df1.columns[7:-3]].filter(like='brand').sum(axis=1)
lob_df1['lob_cineclub'] = df1[df1.columns[7:-3]].filter(like='cineclub').sum(axis=1)
lob_df1['lob_fnb'] = df1[df1.columns[7:-3]].filter(like='fnb').sum(axis=1)

lob_df2['lob_brand'] = df2[df2.columns[7:-3]].filter(like='brand').sum(axis=1)
lob_df2['lob_enhanced'] = df2[df2.columns[7:-3]].filter(like='enhanced').sum(axis=1)
lob_df2['lob_exhibition'] = df2[df2.columns[7:-3]].filter(like='exhibition').sum(axis=1)
lob_df2['lob_film'] = df2[df2.columns[7:-3]].filter(like='film').sum(axis=1)
lob_df2['lob_gne'] = df2[df2.columns[7:-3]].filter(like='gne').sum(axis=1)
lob_df2['lob_prepaid'] = df2[df2.columns[7:-3]].filter(like='prepaid').sum(axis=1)

lob_df3['lob_brand'] = df3[df3.columns[7:-3]].filter(like='brand').sum(axis=1)
lob_df3['lob_cineclub'] = df3[df3.columns[7:-3]].filter(like='cineclub').sum(axis=1)
lob_df3['lob_enhanced'] = df3[df3.columns[7:-3]].filter(like='enhanced').sum(axis=1)
lob_df3['lob_event'] = df3[df3.columns[7:-3]].filter(like='event').sum(axis=1)
lob_df3['lob_exhibition'] = df3[df3.columns[7:-3]].filter(like='exhibition').sum(axis=1)
lob_df3['lob_film'] = df3[df3.columns[7:-3]].filter(like='film').sum(axis=1)
lob_df3['lob_performance'] = df3[df3.columns[7:-3]].filter(like='performance').sum(axis=1)

lob_df1['attendance'] = df1['attendance']
lob_df2['attendance'] = df1['attendance']
lob_df3['attendance'] = df1['attendance']

In [0]:
lob_X1 = lob_df1[lob_df1.columns[:-1]]
lob_X2 = lob_df2[lob_df2.columns[:-1]]
lob_X3 = lob_df3[lob_df3.columns[:-1]]

lob_y = lob_df1['attendance']

In [0]:
# Setting assumptions
custom_config = {
    # Non-negative baseline sales
    'intercept': Prior('Exponential', lam=1),
    # Setting a non-negative control effect
    'gamma_control': Prior('HalfNormal', sigma=Prior('Gamma', alpha=4, beta=1)),
    # Setting an average retention rate of 40% 
    'adstock_alpha': Prior('Beta', alpha=2, beta=3),
    # Setting a moderate ceiling for spend
    'saturation_lam': Prior('Gamma', alpha=4, beta=1)
}

lob_mmm1 = MMM(model_config=custom_config,
          date_column='week',
          channel_columns=lob_X1.columns[7:],
          control_columns=lob_X1.columns[1:7],
          # Maximum effect over 3 weeks
          adstock=GeometricAdstock(l_max=3),
          saturation=LogisticSaturation())

lob_mmm2 = MMM(model_config=custom_config,
          date_column='week',
          channel_columns=lob_X2.columns[7:],
          control_columns=lob_X2.columns[1:7],
          # Maximum effect over 3 weeks
          adstock=GeometricAdstock(l_max=3),
          saturation=LogisticSaturation())

lob_mmm3 = MMM(model_config=custom_config,
          date_column='week',
          channel_columns=lob_X3.columns[7:],
          control_columns=lob_X3.columns[1:7],
          # Maximum effect over 3 weeks
          adstock=GeometricAdstock(l_max=3),
          saturation=LogisticSaturation()) 

lob_mmm1.fit(lob_X1, lob_y, chains=2, random_seed=12, target_accept=0.95)
lob_mmm2.fit(lob_X2, lob_y, chains=2, random_seed=12, target_accept=0.95)
lob_mmm3.fit(lob_X3, lob_y, chains=2, random_seed=12, target_accept=0.95)

lob_mmm1.sample_prior_predictive(lob_X1, extend_idata=True, combined=True)
lob_mmm2.sample_prior_predictive(lob_X2, extend_idata=True, combined=True)
lob_mmm3.sample_prior_predictive(lob_X3, extend_idata=True, combined=True)
lob_mmm1.sample_posterior_predictive(lob_X1, extend_idata=True, combined=True)
lob_mmm2.sample_posterior_predictive(lob_X2, extend_idata=True, combined=True)
lob_mmm3.sample_posterior_predictive(lob_X3, extend_idata=True, combined=True)

lob_y_pred1 = lob_mmm1.predict(lob_X1)
lob_y_pred2 = lob_mmm2.predict(lob_X2)
lob_y_pred3 = lob_mmm3.predict(lob_X3)

In [0]:
fig, ax = plt.subplots(figsize=(12, 8), dpi=120)

# Plotting model
plt.plot(lob_X1['week'], lob_y_pred1, color='tab:blue', label='Predicted Attendance (Awareness)')
plt.plot(lob_X1['week'], lob_y_pred2, color='tab:orange', label='Predicted Attendance (Consideration)')
plt.plot(lob_X1['week'], lob_y_pred3, color='tab:green', label='Predicted Attendance (Conversion)')
plt.plot(lob_X1['week'], lob_y, color='grey', alpha=0.5, label='Actual Attendance')

# Changing aesthetics
plt.ylim(0, 3*10**6)
plt.xlabel('Week')
plt.ylabel('Attendance')
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper right')

plt.show()

In [0]:
lob_mmm1.plot_waterfall_components_decomposition().set_size_inches(12, 8);

In [0]:
lob_mmm2.plot_waterfall_components_decomposition().set_size_inches(12, 8);

In [0]:
lob_mmm3.plot_waterfall_components_decomposition().set_size_inches(12, 8);

# Marketing Mix Model (Aggregated)

In [0]:
# Define data path
path = '/Volumes/training_feature_development/marketing_mix_modelling/data_v1/'

# Load model data for awareness, consideration, and conversion
df1 = pd.read_csv(path + 'model_data/exh_awareness_mmm.csv')
df2 = pd.read_csv(path + 'model_data/exh_consideration_mmm.csv')
df3 = pd.read_csv(path + 'model_data/exh_conversion_mmm.csv')
attendance_and_revenue = pd.read_csv(path + 'clean_data/attendance_and_revenue.csv')

# Select relevant columns from df1 and create a new DataFrame
df = df1[list(df1.columns[:7])+[df1.columns[11]]].copy()

# Calculate total spend for each funnel stage
df['awareness_spend'] = df1[df1.columns[7:-4]].sum(axis=1)
df['consideration_spend'] = df2[df2.columns[7:-4]].sum(axis=1)
df['conversion_spend'] = df3[df3.columns[7:-4]].sum(axis=1)

# Add and transforming columns
df['attendance'] = df1['attendance']
df['total_spend'] = 1.1*(df['awareness_spend'] + df['consideration_spend'] + df['conversion_spend'] + df['pr_spend'])
df['week'] = pd.to_datetime(df['week'])

# Filter for 2024 data only
df = df[df['week'].dt.year==2024]

In [0]:
X = df[list(df.columns[:8])+[df.columns[-1]]]
y = df['attendance']

custom_config = {
    # Non-negative baseline sales
    'intercept': Prior('Exponential', lam=2),
    # Setting a non-negative control effect
    'gamma_control': Prior('HalfNormal', sigma=4),
    # Setting an average retention rate of 40% 
    'adstock_alpha': Prior('Beta', alpha=2, beta=3),
    # Setting a moderate ceiling for spend
    'saturation_lam': Prior('Gamma', alpha=4, beta=1)
}

model = MMM(
    model_config=custom_config,
    date_column='week',
    channel_columns=['total_spend'],
    control_columns=X.columns[1:8],
    # Maximum effect over 3 weeks
    adstock=GeometricAdstock(l_max=3),
    saturation=LogisticSaturation()
)

model.fit(X, y, chains=2, draws=500, random_seed=12, target_accept=0.95)

model.sample_prior_predictive(X, extend_idata=True, combined=True)
model.sample_posterior_predictive(X, extend_idata=True, combined=True)

y_pred = model.predict(X)

In [0]:
plt.figure(figsize=(12, 8))

# Plotting model
plt.plot(y_pred, label="Actual 2024 Attendance")
plt.plot(y.reset_index(drop=True), label="Predicted 2024 Attendance")

# Changing aesthetics
plt.xlabel('Week')
plt.ylabel('Attendance')
plt.ylim(0, 2*10**6)
plt.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.grid(axis='y', alpha=0.25)
plt.legend(loc='upper left')
plt.show()

r2_score(y, y_pred)

In [0]:
model.plot_waterfall_components_decomposition();

In [0]:
# Plotting out the cost curve
fig = model.plot_channel_contribution_grid(start=0.5, stop=5, num=29, absolute_xrange=True)
for collection in fig.gca().collections:
    collection.remove()
fig.set_size_inches(12, 8)
fig.gca().xaxis.set_major_formatter(mtick.EngFormatter(unit=''))
fig.gca().yaxis.set_major_formatter(mtick.EngFormatter(unit=''))
fig.gca().set_xticks(np.arange(0, 1.9*10**7, 2*10**6))
fig.gca().set_yticks(np.arange(0, 2.1*10**7, 10**6))
fig.gca().set_ylim(0, 2*10**7)
fig.gca().set_xlabel('Annual Budget ($)', fontsize=14)
fig.gca().set_ylabel('Annual Attendance Contribution', fontsize=14)
plt.grid(alpha=0.25)
plt.legend(loc='upper right', bbox_to_anchor=(1, 0.99), fontsize=14)
fig.gca().tick_params(axis='both', which='major', labelsize=14)
fig.gca().title.set_fontsize(18);

In [0]:
# Plotting out the marginal cost curve
x = X['total_spend'].reset_index(drop=True)*52
y = model.compute_mean_contributions_over_time(original_scale=True)['total_spend'].reset_index(drop=True)*52
z = y/x

def mod(x, a, b):
    return a + b * np.log(x)

params, _ = curve_fit(mod, x, y)
a, b = params

x_fit = np.linspace(1.5*10**6, 1.55*10**7, 15)
y_fit = mod(x_fit, a, b)
z_fit = np.gradient(y_fit, x_fit)

plt.figure(figsize=(12, 8), dpi=120)
plt.plot(x_fit, z_fit, marker='o', color='tab:blue')
plt.vlines(x=3.1*10**6, ymin=0, ymax=4, linestyle='--', color='black', label='2024 Budget', linewidth=1)
plt.xticks(np.arange(0, 1.6*10**7, 10**6))
plt.gca().xaxis.set_major_formatter(mtick.EngFormatter(unit=''))
plt.xlim(0, 1.6*10**7)
plt.ylim(0, 3)
plt.title('Marginal Contribution of Marketing Budget')
plt.xlabel('Annual Budget ($)')
plt.ylabel('Marginal Attendance Contribution')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [0]:
print(f'Total 2024 budget: {df["total_spend"].sum():,.0f}')
print(f'Total 2024 attendance: {df["attendance"].sum():,.0f}')